# 课程 4 作业 · 把对话历史变成可检查的程序

共 **100 分：1 道简单题 10 分、2 道中等题各 30 分、1 道开放思考题 30 分**。课堂已经提供了主要思路；这里需要自己组织函数、验证和解释。

作业不要求新增 API 调用。第 4 题引用课堂三次真实调用的记录；其他题完全在本地运行。下面的文字和 token 数均明确标作人工测试输入，不是模型实测。

每个小问都有独立答题槽，请逐项作答。没有证据时写“尚未验证”和准备怎样验证，不把预期结果写成已经发生的事。


## 1 · Python 保存了，不等于已经发出（简单，10 分）

同事写了 `previous_answer = "上一轮的模型回答"`，随后向本课使用的普通 Chat Completions 接口只发送：

```python
messages = [
    {"role": "system", "content": "按提供的活动规则回答，不足就询问。"},
    {"role": "user", "content": "那再加一个8岁的孩子呢？"},
]
```

请分别回答：

**1a（3 分）** 这次请求实际发出了哪些内容？是否包含变量 `previous_answer` 的值？

我的答案：____

**1b（3 分）** 同一个模型名和 API key，是否足以让当前程序自动发送上一轮历史？说明原因。

我的答案：____

**1c（4 分）** 假如模型碰巧给出了正确票价，能否据此证明它记住了上一轮？你还需要检查什么？

我的答案：____


## 2 · 自己写消息构造函数（中等，30 分）

实现 `build_messages(system_text, turns, new_question)`，返回一个**新列表**。

- 第一条是 system。
- `turns` 中每轮按 user、assistant 的顺序加入；内容原样保留。
- 最后一条是新的 user 问题。
- 不修改传入的 `turns`，允许零轮、一轮、多轮历史。

`turns` 是我们自己定义的 Python 数据结构，不是服务商 API 的新字段。函数把它转换成已经学过的 role/content 列表。输入内容可以是普通字符串；不要求处理图片、工具调用或服务端会话。

**2a（20 分）** 完成函数。课堂只处理一轮，这里请用循环支持多轮。


In [ ]:
def build_messages(system_text, turns, new_question):
    # TODO H2a：创建新消息列表、依次添加历史、加入追问，然后返回。
    pass

# 明确的人工测试输入；没有在这里调用模型。
test_turns = [
    {"user": "材料：成人88元，青少年48元。我和16岁妹妹共多少钱？", "assistant": "共136元。"},
    {"user": "再加一位17岁的朋友呢？", "assistant": "共184元。"},
]
print(build_messages("只根据材料回答。", test_turns, "现在一共几人？"))


**2b（10 分）** 自己写至少 **3 条断言**，分别检查：

1. 零轮历史：只剩 system 和当前 user，内容正确。（3 分）
2. 两轮历史：role 顺序为 `system, user, assistant, user, assistant, user`，内容和顺序没有丢失。（4 分）
3. 调用前后的 `test_turns` 相同，没有把追问塞进原来的历史。（3 分）

`assert 条件` 在条件为假时会报错；这正是用来发现程序问题的。不要只断言函数“没有报错”。可以在调用前用 `[dict(turn) for turn in test_turns]` 保存副本，供调用后比较。


In [ ]:
# TODO H2b：至少三条 assert，对应上面的三类检查。


## 3 · 算清楚两轮任务和三次实验（中等，30 分）

以下全部是**人工设计的费用练习数据，不是课堂实测**。A 是首次提问，B 是不带历史的独立对照，C 是带历史的追问。

| 请求 | 输入 tokens | 输出 tokens |
|---|---:|---:|
| A | 400 | 100 |
| B | 30 | 80 |
| C | 620 | 150 |

教学假设价：输入 **3 元 / 百万 tokens**，输出 **9 元 / 百万 tokens**。C 的输入 620 已经包含这次提交的历史。

**3a（12 分）** 独立写费用函数，逐次相加，单位为元。如果任一次结果缺少 `prompt_tokens` 或 `completion_tokens`，返回 `None`，不要用 0 补缺失。

可用的已学写法：`result.get("usage", {})` 在缺少 usage 时得到空字典；`"prompt_tokens" not in usage` 检查键是否缺失。用 `for` 逐次检查并累加即可。


In [ ]:
practice_results = {
    "A": {"usage": {"prompt_tokens": 400, "completion_tokens": 100}},
    "B": {"usage": {"prompt_tokens": 30, "completion_tokens": 80}},
    "C": {"usage": {"prompt_tokens": 620, "completion_tokens": 150}},
}
unknown_usage = {"usage": {"prompt_tokens": 50}}  # 输出用量缺失。

def estimate_cost(results, input_per_million=3.0, output_per_million=9.0):
    # TODO H3a：完整实现用量检查、换算和累加。
    pass

# TODO H3b：调用函数，分别打印 A+C、A+B+C、A+unknown_usage 的结果。


**3b（8 分）** 列出公式、单位和计算结果，不能只贴打印输出。

- 两轮任务 A+C：公式 ____；结果 ____ 元。
- 三次实验 A+B+C：公式 ____；结果 ____ 元。

**3c（5 分）** 为什么不在 C 的 620 个输入 tokens 之外，再加一次 A 的输入或输出作为“历史费”？A 自己那次请求又为什么仍要计费？

我的答案：____

**3d（5 分）** `A + unknown_usage` 应返回什么？为什么本地记录的成功费用不能保证等于控制台账单？至少说明一个课堂讲过的原因。

我的答案：____


## 4 · 面试追问：解释对话历史（开放思考，30 分）

面试官看过你的活动咨询实验，问：“我们的 Python 已经保存了聊天变量，应该自动有记忆。只要加上历史以后答对一次，以后都可以放心用了吧？”

请用 **200–350 字左右** 回复，按下面四个槽分别填写。允许结论与其他同学不同，但必须符合课堂的接口机制，并引用自己的真实实验。无需讨论尚未学过的检索、数据库、部署或框架。

**4a · 结论（6 分）** 当前程序应该怎样组织后续请求？给出明确做法。

我的答案：____

**4b · 机制（8 分）** 解释本地变量、消息列表、当前请求的关系；说清本课使用的接口边界。

我的答案：____

**4c · 实验记录（8 分）** 引用课堂 B、C 的实际回答，各摘一句，再用原始规则说明哪些结论成立、哪些尚不能确认。如果 B 没出错或 C 出错，照实分析。

我的答案：____

**4d · 限制（8 分）** 说明一次成功不能证明什么、继续带历史的费用来自哪里，并设计一个仅改变课堂已有条件的小实验（例如改年龄或人数）。只写准备检查的结果，不要求额外调用，也不要声称已经验证。

我的答案：____
